In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
train_data = pd.read_csv('C:/Users/lujai/Downloads/train.csv')
test_data  = pd.read_csv('C:/Users/lujai/Downloads/test.csv')

In [3]:
test_ids = test_data['Id']

In [4]:
train_data.drop(['Id'], axis=1, inplace=True)
test_data.drop(['Id'],  axis=1, inplace=True)

In [5]:
cols_to_drop = ['Alley', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature']
train_data.drop(cols_to_drop, axis=1, inplace=True)
test_data.drop(cols_to_drop,  axis=1, inplace=True)

In [6]:
cols_to_drop2 = ['1stFlrSF', 'TotRmsAbvGrd', 'GarageCars', 'GarageYrBlt']
train_data.drop(cols_to_drop2, axis=1, inplace=True)
test_data.drop(cols_to_drop2,  axis=1, inplace=True)

In [7]:
y = train_data['SalePrice']
train_data.drop(['SalePrice'], axis=1, inplace=True)

In [8]:
train_data = pd.get_dummies(train_data)
test_data  = pd.get_dummies(test_data)

In [9]:
train_data, test_data = train_data.align(test_data, join='left', axis=1, fill_value=0)

In [10]:
imputer = KNNImputer()
train_data = pd.DataFrame(imputer.fit_transform(train_data), columns=train_data.columns)
test_data  = pd.DataFrame(imputer.transform(test_data),      columns=test_data.columns)

In [11]:
X_clean = train_data.loc[:, train_data.nunique() > 1]
X_clean = X_clean.T.drop_duplicates().T

In [12]:
vif_data = pd.DataFrame()
vif_data["Feature"] = X_clean.columns
vif_data["VIF"]     = [variance_inflation_factor(X_clean.values, i) for i in range(X_clean.shape[1])]
print("VIF Results:")
print(vif_data.sort_values("VIF", ascending=False))

c:\Users\lujai\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


VIF Results:
                   Feature       VIF
263  SaleCondition_Partial       inf
262   SaleCondition_Normal       inf
261   SaleCondition_Family       inf
260   SaleCondition_Alloca       inf
243          GarageCond_Gd       inf
..                     ...       ...
156      MasVnrType_BrkCmn  1.335314
27             ScreenPorch  1.321507
31                  YrSold  1.319838
30                  MoSold  1.227133
26               3SsnPorch  1.213005

[264 rows x 2 columns]


In [13]:
good_features = vif_data[vif_data["VIF"] < 10]["Feature"].tolist()
X = train_data[good_features]
test_data = test_data[good_features]

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [15]:
# RANDOM FOREST 
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_rmse  = np.sqrt(mean_squared_error(y_test, rf_preds))
print(f"\nRandom Forest RMSE: {rf_rmse:.2f}")


Random Forest RMSE: 33464.79


In [16]:
# GRADIENT BOOSTING 
gb = GradientBoostingRegressor(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train, y_train)
gb_preds = gb.predict(X_test)
gb_rmse  = np.sqrt(mean_squared_error(y_test, gb_preds))
print(f"Gradient Boosting RMSE: {gb_rmse:.2f}")

Gradient Boosting RMSE: 33019.79


In [17]:
rf_final = rf.predict(test_data)
gb_final = gb.predict(test_data)

In [18]:
submission_rf = pd.DataFrame({'Id': test_ids, 'SalePrice': rf_final})
submission_rf.to_csv('C:/Users/lujai/Downloads/submission_rf.csv', index=False)
print("\nRandom Forest Submission:")
print(submission_rf.head())


Random Forest Submission:
     Id  SalePrice
0  1461  132403.00
1  1462  153811.35
2  1463  167113.00
3  1464  181731.64
4  1465  204928.79


In [19]:
submission_gb = pd.DataFrame({'Id': test_ids, 'SalePrice': gb_final})
submission_gb.to_csv('C:/Users/lujai/Downloads/submission_gb.csv', index=False)
print("\nGradient Boosting Submission:")
print(submission_gb.head())


Gradient Boosting Submission:
     Id      SalePrice
0  1461  133983.831253
1  1462  149190.628207
2  1463  179558.920656
3  1464  188145.939713
4  1465  200956.901052
